In [1]:
import pandas as pd
from transformers import T5Tokenizer , Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
# randoom sampling 
train_data = train_data.sample(n = 4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n =500, random_state=42).reset_index(drop=True)

In [7]:
train_data.shape

(4000, 3)

In [8]:
val_data.shape

(500, 3)

In [9]:
import re

def clean_data (text):
    text = re.sub(r"\r\n", " ",text) #lines
    text = re.sub(r"\s+"," ", text) # spaces
    text = re.sub(r"<.*?>"," ",text) # html tags <p> <h1> all
    text = text.strip().lower()
    
    return text

In [10]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [11]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

In [12]:
# Tokenizer 
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [14]:
# Raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)
    
    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [15]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [16]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
# input ids = dialogue => token ids

# 1 => end of sequance . 0 => padding

# attention mask

# labels - target => summary token 

In [18]:
len(train_dataset[0]["input_ids"])

512

In [19]:
# working with our model 

# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [31]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print("device: ", device)
model.to(device)    

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training Arguments

training_args = TrainingArguments(
    output_dir="./results",
    
    num_train_epochs=6,
    weight_decay=0.01,
    
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    
    warmup_steps=500
    # 0 => lr default
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
# train the model

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.631273,0.379161
2,0.396137,0.360677


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
trainer.train(resume_from_checkpoint=True)

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss
3,0.373064,0.355531
4,0.362364,0.351454
5,0.354644,0.350468
6,0.351133,0.349856


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.24020077514648439, metrics={'train_runtime': 1904.316, 'train_samples_per_second': 12.603, 'train_steps_per_second': 1.575, 'total_flos': 3248203235328000.0, 'train_loss': 0.24020077514648439, 'epoch': 6.0})

In [23]:
# model load => fine-tune => save the model

In [ ]:
# how to save the model..
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [26]:
# how to use the saved model

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [32]:
## test the summerizing model logic

def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # cleaning the user gievn data
    
    # tokenize the Given input from user
    inputs = tokenizer(
        dialogue, # Data 
        padding="max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    ).to(device)
    
    # Genrate the summary => token ids (generating the result)
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4, # generate 4 result and give the best from it.
        early_stopping = True 
    )
    
    # decode our output (decode the model out put to undestand to user)
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [33]:
test_dialoge =""" User: Play my morning playlist
Jarvis: Accessing memory module. Found previous preferences.
Jarvis: You usually play your morning playlist at this time.
Jarvis: I will open the music app, play your playlist, and set volume to your preferred level.
Jarvis: Do you want me to proceed?
Jarvis: Action Plan - Open Music App, Play Playlist, Set Volume"""

summary = summarize_dialogue(test_dialoge)
print("Summary: ", summary)

Summary:  jarvis will open the music app, play playlist, set volume to his preferred level.
